# A2: Laboratory Feature Reduction

## 1. Objective and Experimental Design

This experiment evaluates whether the laboratory feature space can be reduced while retaining the ATC3 medication representation established in A1.

The A1 dataset is used as the starting point. Consequently, the patient cohort, medication representation, routine chart variables, outcome definition, and downstream evaluation framework remain unchanged. Laboratory features are reduced using an outcome-independent criterion defined before model evaluation, separating the effect of laboratory feature reduction from the preceding medication feature-reduction experiment.

The experimental progression is:

- **A0:** reproduced baseline with the original raw medication representation;
- **A1:** raw medication features replaced by ATC3 therapeutic categories; and
- **A2:** the ATC3 representation retained while reducing laboratory features.

No predictive model is used to select laboratory variables.


In [ ]:
import os
import pandas as pd
import numpy as np

PROJECT_DIR = os.environ.get("MIMIC_FAIRNESS_WORKDIR")

if not PROJECT_DIR:
    raise RuntimeError(
        "Set MIMIC_FAIRNESS_WORKDIR to your private research workspace."
    )
DATA_DIR = os.path.join(PROJECT_DIR, "data")

A1_DIR = os.path.join(DATA_DIR, "medication_feature_reduction")
A2_DIR = os.path.join(DATA_DIR, "laboratory_feature_reduction")
os.makedirs(A2_DIR, exist_ok=True)

A1_PARQUET = os.path.join(
    A1_DIR,
    "icu_data_no_imputation_atc3_reduced.parquet"
)

icu_data = pd.read_parquet(A1_PARQUET)

required_columns = [
    "subject_id",
    "stay_id",
    "los",
    "age",
    "gender",
    "insurance",
    "race",
]

assert all(col in icu_data.columns for col in required_columns)
assert len(icu_data) == 74501
assert icu_data["stay_id"].nunique() == 74501

atc3_columns = [col for col in icu_data.columns if col.startswith("med_atc3_")]
assert len(atc3_columns) == 192

print("A2 starting dataset")
print("-------------------")
print("Shape:", icu_data.shape)
print("ICU stays:", icu_data["stay_id"].nunique())
print("Patients:", icu_data["subject_id"].nunique())
print("ATC3 medication features:", len(atc3_columns))


## 2. Laboratory Feature Identification

Laboratory variables are identified from the laboratory matrix generated during the original No-Imputation preprocessing. This matrix provides the feature set before integration with medication variables.

Column-name suffixes introduced during dataset merging are resolved against the original laboratory values rather than inferred from column names alone. This preserves the exact laboratory representation used in A1 and avoids misclassifying medication or chart variables as laboratory features.


In [ ]:
# Laboratory feature identification

LAB_MATRIX = os.path.join(DATA_DIR, "final_labs_no_imputation_reproduction.parquet")
LAB_COLUMN_MAP = os.path.join(A2_DIR, "laboratory_column_mapping.csv")

assert os.path.exists(LAB_MATRIX), f"Laboratory matrix not found: {LAB_MATRIX}"

lab_matrix = pd.read_parquet(LAB_MATRIX)
id_columns = ["subject_id", "stay_id"]
source_lab_columns = [c for c in lab_matrix.columns if c not in id_columns]

print("Laboratory matrix shape:", lab_matrix.shape)
print("Source laboratory features:", len(source_lab_columns))

assert len(source_lab_columns) == 458
# Restrict the laboratory matrix to the final A1 cohort.

cohort_ids = icu_data[id_columns].copy()
cohort_labs = cohort_ids.merge(lab_matrix, on=id_columns, how="left", validate="one_to_one")

assert len(cohort_labs) == len(icu_data)
assert cohort_labs[source_lab_columns].isna().sum().sum() == 0

# Map source laboratory names to their corresponding A1 columns.

a1_columns = set(icu_data.columns)
lab_mapping = []

for lab in source_lab_columns:
    candidates = [c for c in (lab, f"{lab}_x", f"{lab}_y") if c in a1_columns]

    if len(candidates) == 1:
        mapped = candidates[0]
        method = "direct" if mapped == lab else "suffix"

    elif len(candidates) > 1:
        source_values = cohort_labs[lab].to_numpy()
        matches = [
            c for c in candidates
            if np.allclose(source_values, icu_data[c].to_numpy(), equal_nan=True)
        ]

        mapped = matches[0] if len(matches) == 1 else None
        method = "value_match" if mapped is not None else "unresolved"

    else:
        mapped = None
        method = "unresolved"

    lab_mapping.append({
        "source_lab": lab,
        "a1_column": mapped,
        "mapping_method": method
    })

lab_mapping = pd.DataFrame(lab_mapping)

# Validate the laboratory feature set.

unresolved_labs = lab_mapping[lab_mapping["a1_column"].isna()]
lab_columns = lab_mapping["a1_column"].dropna().tolist()

print()
print("Laboratory feature identification")
print("---------------------------------")
print("Source laboratory features:", len(source_lab_columns))
print("Mapped laboratory features:", len(lab_columns))
print("Unique mapped columns:", len(set(lab_columns)))
print("Direct mappings:", (lab_mapping["mapping_method"] == "direct").sum())
print("Suffix mappings:", (lab_mapping["mapping_method"] == "suffix").sum())
print("Value-resolved mappings:", (lab_mapping["mapping_method"] == "value_match").sum())
print("Unresolved features:", len(unresolved_labs))
print("Overlap with ATC3 features:", len(set(lab_columns) & set(atc3_columns)))

assert len(unresolved_labs) == 0
assert len(lab_columns) == 458
assert len(set(lab_columns)) == 458
assert len(set(lab_columns) & set(atc3_columns)) == 0

lab_mapping.to_csv(LAB_COLUMN_MAP, index=False)
print("Mapping saved:", LAB_COLUMN_MAP)


## 3. Laboratory Coverage and Sparsity Audit

Laboratory feature reduction is based on observation frequency rather than predictive association with the LOS outcome.

Because the No-Imputation representation replaces unobserved laboratory measurements with zero, missingness cannot be inferred directly from the wide model matrix. Observation coverage is therefore calculated from the processed long-format laboratory data before zero filling.

For each laboratory variable, the audit records the number and proportion of ICU stays with at least one numeric observation, the number of numeric events, and descriptive properties of the corresponding zero-filled model feature. No laboratory variable is removed during the audit.


In [ ]:
import duckdb

MIMIC_DIR = os.path.join(PROJECT_DIR, "mimic-iv-3.1")
PROCESSED_LABS = os.path.join(DATA_DIR, "filtered_labs_processed.parquet")
D_LABITEMS = os.path.join(MIMIC_DIR, "hosp", "d_labitems.csv.gz")
LAB_AUDIT_CSV = os.path.join(A2_DIR, "laboratory_feature_audit.csv")

assert os.path.exists(PROCESSED_LABS)
assert os.path.exists(D_LABITEMS)

processed_labs_path = PROCESSED_LABS.replace("\\", "/")
labitems_path = D_LABITEMS.replace("\\", "/")

cohort_ids = icu_data[["subject_id", "stay_id"]].drop_duplicates()

con = duckdb.connect()
con.register("cohort_ids", cohort_ids)

coverage = con.execute(f"""
    SELECT
        d.label AS source_lab,
        COUNT(DISTINCT CASE
            WHEN TRY_CAST(l.value AS DOUBLE) IS NOT NULL
            THEN l.stay_id
        END) AS observed_stays,
        COUNT(CASE
            WHEN TRY_CAST(l.value AS DOUBLE) IS NOT NULL
            THEN 1
        END) AS numeric_events
    FROM read_parquet('{processed_labs_path}') AS l
    INNER JOIN cohort_ids AS c
        ON l.subject_id = c.subject_id
       AND l.stay_id = c.stay_id
    INNER JOIN read_csv_auto('{labitems_path}', header=true) AS d
        ON l.itemid = d.itemid
    WHERE d.label IS NOT NULL
    GROUP BY d.label
""").fetchdf()

con.close()

audit = lab_mapping.merge(coverage, on="source_lab", how="left")
audit["observed_stays"] = audit["observed_stays"].fillna(0).astype(int)
audit["numeric_events"] = audit["numeric_events"].fillna(0).astype(int)

n_stays = len(icu_data)
audit["coverage_rate"] = audit["observed_stays"] / n_stays
audit["unobserved_rate"] = 1 - audit["coverage_rate"]

model_stats = []
for row in audit.itertuples():
    values = pd.to_numeric(icu_data[row.a1_column], errors="coerce")
    model_stats.append({
        "source_lab": row.source_lab,
        "model_zero_rate": (values == 0).mean(),
        "model_variance": values.var(ddof=0),
        "model_mean": values.mean(),
    })

model_stats = pd.DataFrame(model_stats)
audit = audit.merge(model_stats, on="source_lab", how="left")

audit = audit[
    [
        "source_lab",
        "a1_column",
        "mapping_method",
        "observed_stays",
        "coverage_rate",
        "unobserved_rate",
        "numeric_events",
        "model_zero_rate",
        "model_mean",
        "model_variance",
    ]
].sort_values("coverage_rate", ascending=False)

assert len(audit) == 458
assert audit["a1_column"].nunique() == 458
assert audit["coverage_rate"].between(0, 1).all()


### 3.1 Resolution of duplicated laboratory labels

The wide laboratory matrix contains one automatically suffixed feature, `Mesothelial cells_1`. This suffix results from laboratory labels that differ only in capitalization (`Mesothelial Cells` and `Mesothelial cells`) and does not represent a distinct source label.

Coverage for the suffixed feature is therefore reconstructed from its corresponding source-label observations using the same numeric-value and stay-level aggregation logic as the original laboratory matrix.


In [ ]:
# Resolve the case-sensitive source labels represented by the suffixed pivot column.
d_labitems = pd.read_csv(
    D_LABITEMS,
    usecols=["itemid", "label"],
    low_memory=False,
)

target_labels = ["Mesothelial Cells", "Mesothelial cells"]
target_columns = ["Mesothelial Cells", "Mesothelial cells_1"]

label_items = d_labitems[
    d_labitems["label"].isin(target_labels)
][["itemid", "label"]].drop_duplicates()

# Retrieve only the relevant numeric events in the final ICU cohort.
con = duckdb.connect()
con.register("cohort_ids", cohort_ids)
con.register("label_items", label_items)

events = con.execute(f"""
    SELECT
        l.subject_id,
        l.stay_id,
        li.label,
        TRY_CAST(l.value AS DOUBLE) AS numeric_value
    FROM read_parquet('{processed_labs_path}') AS l
    INNER JOIN cohort_ids AS c
        ON l.subject_id = c.subject_id
       AND l.stay_id = c.stay_id
    INNER JOIN label_items AS li
        ON l.itemid = li.itemid
    WHERE TRY_CAST(l.value AS DOUBLE) IS NOT NULL
""").fetchdf()

con.close()

reconstructed = (
    events.groupby(["subject_id", "stay_id", "label"], as_index=False)["numeric_value"]
    .mean()
)

label_to_column = {}
for label in target_labels:
    expected = cohort_ids.merge(
        reconstructed[reconstructed["label"] == label][
            ["subject_id", "stay_id", "numeric_value"]
        ],
        on=["subject_id", "stay_id"],
        how="left",
    )["numeric_value"].fillna(0).to_numpy()

    matches = [
        column
        for column in target_columns
        if np.allclose(
            expected,
            cohort_labs[column].to_numpy(),
            equal_nan=True,
        )
    ]

    assert len(matches) == 1
    label_to_column[label] = matches[0]

# Correct the audit entry associated with the automatically suffixed feature.
for label, column in label_to_column.items():
    if column.endswith("_1"):
        label_events = events[events["label"] == label]
        observed_stays = label_events["stay_id"].nunique()
        numeric_events = len(label_events)

        mask = audit["a1_column"] == column
        audit.loc[mask, "observed_stays"] = observed_stays
        audit.loc[mask, "numeric_events"] = numeric_events
        audit.loc[mask, "coverage_rate"] = observed_stays / n_stays
        audit.loc[mask, "unobserved_rate"] = 1 - observed_stays / n_stays

audit = audit.sort_values("coverage_rate", ascending=False)
audit.to_csv(LAB_AUDIT_CSV, index=False)

corrected = audit[audit["a1_column"] == "Mesothelial cells_1"]

print("Resolved duplicate-label mapping:", label_to_column)
print()
print("Corrected laboratory feature")
display(
    corrected[
        [
            "source_lab",
            "a1_column",
            "observed_stays",
            "coverage_rate",
            "numeric_events",
            "model_zero_rate",
            "model_variance",
        ]
    ]
)


In [ ]:
thresholds = [0.01, 0.05, 0.10, 0.20, 0.30, 0.50, 0.75, 0.90]

threshold_summary = pd.DataFrame({
    "minimum_coverage": thresholds,
    "features_retained": [
        int((audit["coverage_rate"] >= threshold).sum())
        for threshold in thresholds
    ],
})

threshold_summary["features_removed"] = (
    len(audit) - threshold_summary["features_retained"]
)

print("Final laboratory coverage audit")
print("-------------------------------")
print("Laboratory features:", len(audit))
print("Median coverage:", f"{audit['coverage_rate'].median():.2%}")
print("Mean coverage:", f"{audit['coverage_rate'].mean():.2%}")
print()
print("Feature counts by minimum coverage")
display(threshold_summary)


## 4. Selection of the Laboratory Coverage Threshold

Candidate thresholds are evaluated using feature availability only, without reference to LOS labels or predictive performance.

A 5% minimum-coverage threshold retains 120 laboratory variables, whereas a 10% threshold retains 99. The difference between these candidate representations is therefore limited to 21 laboratory variables.

To determine whether the more conservative 5% threshold retains clinically meaningful information rather than predominantly sparse measurements, the laboratory variables occurring in 5--10% of ICU stays are examined explicitly. This assessment precedes model training and does not use the outcome variable.


In [ ]:
COVERAGE_5 = 0.05
COVERAGE_10 = 0.10

labs_5 = audit[audit["coverage_rate"] >= COVERAGE_5].copy()
labs_10 = audit[audit["coverage_rate"] >= COVERAGE_10].copy()

additional_5 = audit[
    (audit["coverage_rate"] >= COVERAGE_5) &
    (audit["coverage_rate"] < COVERAGE_10)
].copy().sort_values("coverage_rate", ascending=False)

print("Candidate laboratory representations")
print("------------------------------------")
print("Original laboratory features:", len(audit))
print("Retained at >= 5% coverage:", len(labs_5))
print("Retained at >=10% coverage:", len(labs_10))
print("Additional features retained by 5%:", len(additional_5))

assert len(labs_5) == 120
assert len(labs_10) == 99
assert len(additional_5) == 21

display(
    additional_5[
        [
            "source_lab",
            "observed_stays",
            "coverage_rate",
            "numeric_events",
            "model_zero_rate",
            "model_variance"
        ]
    ].reset_index(drop=True)
)

candidate_summary = pd.DataFrame({
    "criterion": [">=5% coverage", ">=10% coverage"],
    "laboratory_features": [len(labs_5), len(labs_10)],
    "median_coverage": [
        labs_5["coverage_rate"].median(),
        labs_10["coverage_rate"].median()
    ],
    "mean_coverage": [
        labs_5["coverage_rate"].mean(),
        labs_10["coverage_rate"].mean()
    ],
    "median_model_zero_rate": [
        labs_5["model_zero_rate"].median(),
        labs_10["model_zero_rate"].median()
    ]
})

print("Candidate-set summary")
display(candidate_summary)

THRESHOLD_COMPARISON = os.path.join(
    A2_DIR,
    "laboratory_5pct_vs_10pct_comparison.csv"
)

additional_5.to_csv(THRESHOLD_COMPARISON, index=False)
print("Comparison saved:", THRESHOLD_COMPARISON)


## 5. Laboratory Feature Selection

A minimum observation coverage of 5% is adopted for the A2 laboratory representation.

This threshold retains laboratory variables with at least one numeric observation in 5% or more of the 74,501 ICU stays. The criterion is defined independently of the LOS outcome and before model evaluation.

The 5% threshold retains 120 of the original 458 laboratory variables. A more restrictive 10% threshold would retain 99 variables, providing only a further reduction of 21 features. Because several variables occurring between 5% and 10% coverage represent clinically interpretable measurements, the 5% threshold is used as the more conservative reduction criterion.

No laboratory variable is selected or removed according to its association with LOS or downstream model performance.


In [ ]:
# Final A2 laboratory feature set

LAB_COVERAGE_THRESHOLD = 0.05

selected_labs = (
    audit[audit["coverage_rate"] >= LAB_COVERAGE_THRESHOLD]
    .copy()
    .sort_values("coverage_rate", ascending=False)
)

selected_lab_columns = selected_labs["a1_column"].tolist()
removed_lab_columns = [
    c for c in lab_columns
    if c not in selected_lab_columns
]

assert len(selected_lab_columns) == 120
assert len(removed_lab_columns) == 338
assert len(set(selected_lab_columns) & set(removed_lab_columns)) == 0
assert set(selected_lab_columns) | set(removed_lab_columns) == set(lab_columns)

SELECTED_LABS_CSV = os.path.join(
    A2_DIR,
    "selected_laboratory_features_5pct.csv"
)

selected_labs.to_csv(SELECTED_LABS_CSV, index=False)

print("A2 laboratory selection")
print("-----------------------")
print("Coverage threshold:", f"{LAB_COVERAGE_THRESHOLD:.0%}")
print("Original laboratory features:", len(lab_columns))
print("Retained laboratory features:", len(selected_lab_columns))
print("Removed laboratory features:", len(removed_lab_columns))
print(
    "Laboratory dimensionality reduction:",
    f"{100 * len(removed_lab_columns) / len(lab_columns):.2f}%"
)
print("Selection saved:", SELECTED_LABS_CSV)


## 6. Construction of the A2 Feature Dataset

The A2 dataset is constructed from the completed A1 representation. Only laboratory variables with observation coverage below 5% are removed.

The ATC3 medication representation, routine chart variables, demographic information, identifiers, LOS outcome information, and patient cohort are retained unchanged.


In [ ]:
A2_PARQUET = os.path.join(
    A2_DIR,
    "icu_data_no_imputation_atc3_lab5pct.parquet"
)

A2_CSV = os.path.join(
    A2_DIR,
    "icu_data_no_imputation_atc3_lab5pct.csv"
)

a2_data = icu_data.drop(columns=removed_lab_columns).copy()

assert len(a2_data) == len(icu_data)
assert a2_data["stay_id"].nunique() == icu_data["stay_id"].nunique()
assert a2_data["subject_id"].nunique() == icu_data["subject_id"].nunique()

assert all(c in a2_data.columns for c in selected_lab_columns)
assert not any(c in a2_data.columns for c in removed_lab_columns)
assert all(c in a2_data.columns for c in atc3_columns)

a2_data.to_parquet(A2_PARQUET, index=False)
a2_data.to_csv(A2_CSV, index=False)

print("A2 dataset")
print("----------")
print("Shape:", a2_data.shape)
print("ICU stays:", a2_data["stay_id"].nunique())
print("Patients:", a2_data["subject_id"].nunique())
print("ATC3 medication features:", len(atc3_columns))
print("Laboratory features:", len(selected_lab_columns))
print("Laboratory features removed:", len(removed_lab_columns))
print()
print("Saved:", A2_PARQUET)
print("Saved:", A2_CSV)


## 7. Patient-Level Split Integrity

The A2 dataset must retain the same patient-level training and held-out partitions used in A1. This ensures that subsequent performance differences can be attributed to laboratory feature reduction rather than changes in cohort composition.

The original two-stage patient-level splitting procedure is reproduced using the same random seed. Split identities derived independently from A1 and A2 are then compared directly.


In [ ]:
from sklearn.model_selection import train_test_split

# Preserve the outcome definition used in the reproduced baseline.
icu_data["los_type"] = np.where(icu_data["los"] > 4, "Long", "Short")
a2_data["los_type"] = np.where(a2_data["los"] > 4, "Long", "Short")

# Outer patient-level split.
a1_patient_ids = icu_data["subject_id"].unique()
a2_patient_ids = a2_data["subject_id"].unique()

a1_train_ids, a1_validation_ids = train_test_split(
    a1_patient_ids, test_size=0.2, random_state=42
)
a2_train_ids, a2_validation_ids = train_test_split(
    a2_patient_ids, test_size=0.2, random_state=42
)

a1_train_data = icu_data[icu_data["subject_id"].isin(a1_train_ids)].copy()
a1_validation_data = icu_data[icu_data["subject_id"].isin(a1_validation_ids)].copy()

train_data = a2_data[a2_data["subject_id"].isin(a2_train_ids)].copy()
validation_data = a2_data[a2_data["subject_id"].isin(a2_validation_ids)].copy()

# Inner patient-level split, following the original implementation exactly.
a1_inner_patient_ids = a1_train_data["subject_id"].unique()
a2_inner_patient_ids = train_data["subject_id"].unique()

a1_train_train_ids, a1_test_train_ids = train_test_split(
    a1_inner_patient_ids, test_size=0.2, random_state=42
)
train_train_ids, test_train_ids = train_test_split(
    a2_inner_patient_ids, test_size=0.2, random_state=42
)

# Canonical A2 split identifiers used by all downstream cells.
train_patient_ids = a2_train_ids
validation_patient_ids = a2_validation_ids

a1_train_train_data = a1_train_data[
    a1_train_data["subject_id"].isin(a1_train_train_ids)
].copy()
a1_test_train_data = a1_train_data[
    a1_train_data["subject_id"].isin(a1_test_train_ids)
].copy()

train_train_data = train_data[
    train_data["subject_id"].isin(train_train_ids)
].copy()
test_train_data = train_data[
    train_data["subject_id"].isin(test_train_ids)
].copy()

split_checks = {
    "Outer training patients identical":
        set(a1_train_ids) == set(a2_train_ids),

    "Outer held-out patients identical":
        set(a1_validation_ids) == set(a2_validation_ids),

    "Inner training patients identical":
        set(a1_train_train_ids) == set(train_train_ids),

    "Inner testing patients identical":
        set(a1_test_train_ids) == set(test_train_ids),
}

outer_leakage = (
    set(train_data["subject_id"]) &
    set(validation_data["subject_id"])
)

inner_leakage = (
    set(train_train_data["subject_id"]) &
    set(test_train_data["subject_id"])
)

print("A1 vs A2 split integrity")
print("------------------------")
for label, result in split_checks.items():
    print(f"{label}: {result}")

print()
print("A2 patient-level split")
print("----------------------")
print("Total rows:", len(a2_data))
print("Total patients:", a2_data["subject_id"].nunique())
print("Outer training rows:", len(train_data))
print("Outer held-out rows:", len(validation_data))
print("Outer training patients:", train_data["subject_id"].nunique())
print("Outer held-out patients:", validation_data["subject_id"].nunique())
print("Inner training rows:", len(train_train_data))
print("Inner testing rows:", len(test_train_data))
print("Inner training patients:", train_train_data["subject_id"].nunique())
print("Inner testing patients:", test_train_data["subject_id"].nunique())
print("Outer patient leakage:", len(outer_leakage))
print("Inner patient leakage:", len(inner_leakage))

assert all(split_checks.values())
assert len(train_data) == 59515
assert len(validation_data) == 14986
assert len(train_train_data) == 47543
assert len(test_train_data) == 11972
assert len(outer_leakage) == 0
assert len(inner_leakage) == 0


## 8. XGBoost Evaluation of the Reduced Laboratory Representation

The A2 feature set is evaluated using the same modeling procedure as A1 and the reproduced baseline.

The patient partitions, outcome definition, non-laboratory features, ATC3 medication representation, feature exclusions, numeric-feature selection, standardization procedure, XGBoost configuration, and decision threshold are retained unchanged. Therefore, differences between A1 and A2 reflect the removal of laboratory variables with less than 5% observation coverage.


In [ ]:
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    confusion_matrix,
    classification_report
)

# Reload the CSV to preserve the modeling dtype behavior used in A1.
model_data = pd.read_csv(A2_CSV, low_memory=False)
model_data["los_type"] = np.where(model_data["los"] > 4, "Long", "Short")

train_data = model_data[
    model_data["subject_id"].isin(train_patient_ids)
].copy()

validation_data = model_data[
    model_data["subject_id"].isin(validation_patient_ids)
].copy()

train_train_data = train_data[
    train_data["subject_id"].isin(train_train_ids)
].copy()

test_train_data = train_data[
    train_data["subject_id"].isin(test_train_ids)
].copy()

assert len(train_train_data) == 47543
assert len(test_train_data) == 11972
assert len(validation_data) == 14986


exclude_columns = [
    "subject_id", "hadm_id", "stay_id",
    "first_careunit", "last_careunit",
    "intime", "outtime", "los", "los_type",
    "age", "gender", "insurance", "race",
    "PHYSICIAN REFERRAL"
]

feature_columns = [
    col for col in train_data.columns
    if col not in exclude_columns
]

numeric_columns = (
    train_train_data[feature_columns]
    .select_dtypes(include=["float64", "int64"])
    .columns
)

selected_atc3_model = [
    c for c in atc3_columns
    if c in numeric_columns
]

selected_lab_model = [
    c for c in selected_lab_columns
    if c in numeric_columns
]

other_model_features = [
    c for c in numeric_columns
    if c not in selected_atc3_model
    and c not in selected_lab_model
]

print("A2 model feature audit")
print("----------------------")
print("Candidate feature columns:", len(feature_columns))
print("Numeric model features:", len(numeric_columns))
print("ATC3 medication features:", len(selected_atc3_model))
print("Laboratory features:", len(selected_lab_model))
print("Other numeric features:", len(other_model_features))


In [ ]:
X_train = train_train_data[numeric_columns]
X_test = test_train_data[numeric_columns]

y_train = train_train_data["los_type"].map({
    "Short": 0,
    "Long": 1
})

y_test = test_train_data["los_type"].map({
    "Short": 0,
    "Long": 1
})

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = XGBClassifier(
    objective="binary:logistic",
    random_state=42
)

model.fit(X_train_scaled, y_train)

y_prob = model.predict_proba(X_test_scaled)[:, 1]
y_pred = (y_prob > 0.5).astype(int)

roc_auc = roc_auc_score(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)
accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print()
print("A2 internal XGBoost evaluation")
print("--------------------------------")
print(f"AUC-ROC: {roc_auc:.4f}")
print(f"AUC-PR:  {pr_auc:.4f}")
print(f"Accuracy: {accuracy:.4f}")
print("Confusion Matrix:")
print(cm)

print()
print("Classification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Short", "Long"],
        digits=4
    )
)


## 9. Held-Out Evaluation

The trained A2 model is evaluated on the same outer held-out patient cohort used in A0 and A1.

No model retraining, feature selection, threshold adjustment, or scaler fitting is performed on the held-out data. The scaler fitted on the inner training partition and the corresponding XGBoost model are applied directly to the held-out observations.


In [ ]:
# Apply the training feature set and fitted scaler to the outer held-out cohort.

X_validation = validation_data[numeric_columns]
y_validation = validation_data["los_type"].map({
    "Short": 0,
    "Long": 1
})

assert len(X_validation) == 14986
assert list(X_validation.columns) == list(numeric_columns)

X_validation_scaled = scaler.transform(X_validation)

validation_prob = model.predict_proba(
    X_validation_scaled
)[:, 1]

validation_pred = (
    validation_prob > 0.5
).astype(int)

validation_roc = roc_auc_score(
    y_validation,
    validation_prob
)

validation_pr = average_precision_score(
    y_validation,
    validation_prob
)

validation_accuracy = accuracy_score(
    y_validation,
    validation_pred
)

validation_cm = confusion_matrix(
    y_validation,
    validation_pred
)

print("A2 held-out XGBoost evaluation")
print("--------------------------------")
print(f"AUC-ROC: {validation_roc:.4f}")
print(f"AUC-PR:  {validation_pr:.4f}")
print(f"Accuracy: {validation_accuracy:.4f}")

print("Confusion Matrix:")
print(validation_cm)

print()
print("Classification Report:")
print(
    classification_report(
        y_validation,
        validation_pred,
        target_names=["Short", "Long"],
        digits=4
    )
)


## 10. Subgroup Performance Evaluation

Subgroup performance is evaluated using the same 14,986 held-out ICU-stay predictions used for the overall A2 evaluation; no subgroup-specific models are trained.

Performance is reported separately by insurance and race using Long-stay precision, recall, F1-score, false-positive rate, and accuracy. These descriptive measures are used to assess whether laboratory feature reduction changes subgroup performance patterns.


In [ ]:
# Construct subgroup evaluation data from the held-out predictions.

fairness_data = pd.DataFrame({
    "Actual": y_validation.to_numpy(),
    "Predicted": validation_pred,
    "Insurance": validation_data["insurance"].to_numpy(),
    "Race": validation_data["race"].to_numpy()
})

assert len(fairness_data) == 14986

def subgroup_summary(data, group_column):
    rows = []

    for group, group_data in data.groupby(group_column, dropna=False):
        y_true = group_data["Actual"]
        y_pred = group_data["Predicted"]

        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()

        precision = tp / (tp + fp) if (tp + fp) else np.nan
        recall = tp / (tp + fn) if (tp + fn) else np.nan
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall
            else np.nan
        )
        fpr = fp / (fp + tn) if (fp + tn) else np.nan

        rows.append({
            group_column: group,
            "n": len(group_data),
            "TN": tn,
            "FP": fp,
            "FN": fn,
            "TP": tp,
            "accuracy": (tp + tn) / len(group_data),
            "long_precision": precision,
            "long_recall": recall,
            "long_f1": f1,
            "TPR": recall,
            "FPR": fpr,
            "positive_prediction_rate": (tp + fp) / len(group_data)
        })

    return (
        pd.DataFrame(rows)
        .sort_values("n", ascending=False)
        .reset_index(drop=True)
    )

insurance_fairness_a2 = subgroup_summary(
    fairness_data.dropna(subset=["Insurance"]),
    "Insurance"
)

race_fairness_a2 = subgroup_summary(
    fairness_data.dropna(subset=["Race"]),
    "Race"
)

print("A2 held-out subgroup performance: Insurance")
display(
    insurance_fairness_a2[
        [
            "Insurance",
            "n",
            "long_precision",
            "long_recall",
            "long_f1",
            "FPR",
            "accuracy"
        ]
    ]
)

print("A2 held-out subgroup performance: Race")
display(
    race_fairness_a2[
        [
            "Race",
            "n",
            "long_precision",
            "long_recall",
            "long_f1",
            "FPR",
            "accuracy"
        ]
    ]
)

insurance_fairness_a2.to_csv(
    os.path.join(A2_DIR, "a2_insurance_subgroup_performance.csv"),
    index=False
)

race_fairness_a2.to_csv(
    os.path.join(A2_DIR, "a2_race_subgroup_performance.csv"),
    index=False
)


## 11. Controlled A1–A2 Comparison

A1 is reconstructed from its saved reduced-medication dataset using the identical patient partitions applied in A2. The A1 feature set, standardization procedure, XGBoost configuration, and 0.5 decision threshold are retained.

This produces a directly reproducible comparison in which the laboratory feature reduction is the only intended change between A1 and A2.


In [ ]:
# Reconstruct the A1 reference model using the identical patient partitions.

A1_CSV = os.path.join(
    A1_DIR,
    "icu_data_no_imputation_atc3_reduced.csv"
)

a1_model_data = pd.read_csv(A1_CSV, low_memory=False)
a1_model_data["los_type"] = np.where(
    a1_model_data["los"] > 4, "Long", "Short"
)

a1_train = a1_model_data[
    a1_model_data["subject_id"].isin(train_patient_ids)
].copy()

a1_inner_train = a1_train[
    a1_train["subject_id"].isin(train_train_ids)
].copy()

a1_inner_test = a1_train[
    a1_train["subject_id"].isin(test_train_ids)
].copy()

a1_validation = a1_model_data[
    a1_model_data["subject_id"].isin(validation_patient_ids)
].copy()

assert len(a1_inner_train) == 47543
assert len(a1_inner_test) == 11972
assert len(a1_validation) == 14986
a1_feature_columns = [
    c for c in a1_train.columns
    if c not in exclude_columns
]

a1_numeric_columns = (
    a1_inner_train[a1_feature_columns]
    .select_dtypes(include=["float64", "int64"])
    .columns
)

assert len(a1_numeric_columns) == 691

X_a1_train = a1_inner_train[a1_numeric_columns]
X_a1_validation = a1_validation[a1_numeric_columns]

y_a1_train = a1_inner_train["los_type"].map({
    "Short": 0,
    "Long": 1
})

y_a1_validation = a1_validation["los_type"].map({
    "Short": 0,
    "Long": 1
})

a1_scaler = StandardScaler()
X_a1_train_scaled = a1_scaler.fit_transform(X_a1_train)
X_a1_validation_scaled = a1_scaler.transform(X_a1_validation)

a1_model = XGBClassifier(
    objective="binary:logistic",
    random_state=42
)

a1_model.fit(X_a1_train_scaled, y_a1_train)

a1_validation_prob = a1_model.predict_proba(
    X_a1_validation_scaled
)[:, 1]

a1_validation_pred = (
    a1_validation_prob > 0.5
).astype(int)

a1_roc = roc_auc_score(
    y_a1_validation,
    a1_validation_prob
)

a1_pr = average_precision_score(
    y_a1_validation,
    a1_validation_prob
)

a1_accuracy = accuracy_score(
    y_a1_validation,
    a1_validation_pred
)

a1_cm = confusion_matrix(
    y_a1_validation,
    a1_validation_pred
)

print("Regenerated A1 held-out reference")
print("---------------------------------")
print("Model features:", len(a1_numeric_columns))
print(f"AUC-ROC: {a1_roc:.4f}")
print(f"AUC-PR:  {a1_pr:.4f}")
print(f"Accuracy: {a1_accuracy:.4f}")
print("Confusion Matrix:")
print(a1_cm)


In [ ]:
a1_fairness_data = pd.DataFrame({
    "Actual": y_a1_validation.to_numpy(),
    "Predicted": a1_validation_pred,
    "Insurance": a1_validation["insurance"].to_numpy(),
    "Race": a1_validation["race"].to_numpy()
})

a1_insurance = subgroup_summary(
    a1_fairness_data.dropna(subset=["Insurance"]),
    "Insurance"
)

a1_race = subgroup_summary(
    a1_fairness_data.dropna(subset=["Race"]),
    "Race"
)

insurance_comparison = a1_insurance.merge(
    insurance_fairness_a2,
    on="Insurance",
    suffixes=("_A1", "_A2")
)

race_comparison = a1_race.merge(
    race_fairness_a2,
    on="Race",
    suffixes=("_A1", "_A2")
)

for df in [insurance_comparison, race_comparison]:
    df["delta_long_recall"] = (
        df["long_recall_A2"] - df["long_recall_A1"]
    )
    df["delta_FPR"] = (
        df["FPR_A2"] - df["FPR_A1"]
    )
    df["delta_long_f1"] = (
        df["long_f1_A2"] - df["long_f1_A1"]
    )

print("A1 vs A2: Insurance")
display(
    insurance_comparison[
        [
            "Insurance",
            "n_A1",
            "long_recall_A1",
            "long_recall_A2",
            "delta_long_recall",
            "FPR_A1",
            "FPR_A2",
            "delta_FPR",
            "long_f1_A1",
            "long_f1_A2",
            "delta_long_f1"
        ]
    ]
)

print("A1 vs A2: Race")
display(
    race_comparison[
        [
            "Race",
            "n_A1",
            "long_recall_A1",
            "long_recall_A2",
            "delta_long_recall",
            "FPR_A1",
            "FPR_A2",
            "delta_FPR",
            "long_f1_A1",
            "long_f1_A2",
            "delta_long_f1"
        ]
    ]
)

insurance_comparison.to_csv(
    os.path.join(A2_DIR, "a1_vs_a2_insurance_comparison.csv"),
    index=False
)

race_comparison.to_csv(
    os.path.join(A2_DIR, "a1_vs_a2_race_comparison.csv"),
    index=False
)


## 12. Experimental Summary

The sequential experiments isolate the effects of two feature-reduction interventions.

- **A0** represents the reproduced baseline using raw medication indicators and the complete No-Imputation laboratory representation.
- **A1** replaces the raw medication indicators with ATC3 therapeutic categories while retaining the original laboratory feature set.
- **A2** retains the ATC3 medication representation and removes laboratory variables observed in fewer than 5% of ICU stays.

The cohort, LOS definition, patient-level partitions, XGBoost configuration, prediction threshold, and subgroup evaluation procedure remain unchanged across the comparisons. Consequently, A0--A1 isolates medication representation, whereas A1--A2 isolates laboratory feature reduction.


In [ ]:
# Verified A0 reference from the reproduced No-Imputation baseline.
a0_features = 5721
a0_roc = 0.8791
a0_pr = 0.7858
a0_accuracy = 0.8312
a0_cm = np.array([[9668, 893], [1636, 2789]])

def binary_metrics_from_cm(cm):
    tn, fp, fn, tp = cm.ravel()
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    f1 = 2 * precision * recall / (precision + recall)
    return precision, recall, f1

a0_precision, a0_recall, a0_f1 = binary_metrics_from_cm(a0_cm)
a1_precision, a1_recall, a1_f1 = binary_metrics_from_cm(a1_cm)
a2_precision, a2_recall, a2_f1 = binary_metrics_from_cm(validation_cm)

overall_summary = pd.DataFrame({
    "Experiment": ["A0", "A1", "A2"],
    "Representation": [
        "Raw medications + 458 labs",
        "192 ATC3 medications + 458 labs",
        "192 ATC3 medications + 120 labs"
    ],
    "Model_features": [
        a0_features,
        len(a1_numeric_columns),
        len(numeric_columns)
    ],
    "ROC_AUC": [
        a0_roc,
        a1_roc,
        validation_roc
    ],
    "PR_AUC": [
        a0_pr,
        a1_pr,
        validation_pr
    ],
    "Accuracy": [
        a0_accuracy,
        a1_accuracy,
        validation_accuracy
    ],
    "Long_precision": [
        a0_precision,
        a1_precision,
        a2_precision
    ],
    "Long_recall": [
        a0_recall,
        a1_recall,
        a2_recall
    ],
    "Long_F1": [
        a0_f1,
        a1_f1,
        a2_f1
    ]
})

display(overall_summary.round(4))


In [ ]:
dimensionality_summary = pd.DataFrame({
    "Comparison": ["A0 → A1", "A1 → A2", "A0 → A2"],
    "Features_before": [5721, 691, 5721],
    "Features_after": [691, 353, 353]
})

dimensionality_summary["Features_removed"] = (
    dimensionality_summary["Features_before"] -
    dimensionality_summary["Features_after"]
)

dimensionality_summary["Reduction_percent"] = (
    100 *
    dimensionality_summary["Features_removed"] /
    dimensionality_summary["Features_before"]
)

print("Feature-space reduction")
display(dimensionality_summary.round(2))


In [ ]:
def group_spread(df, group_column, minimum_n=100):
    eligible = df[df["n"] >= minimum_n]

    return {
        "Groups": len(eligible),
        "TPR_spread": (
            eligible["TPR"].max() -
            eligible["TPR"].min()
        ),
        "FPR_spread": (
            eligible["FPR"].max() -
            eligible["FPR"].min()
        )
    }

insurance_a1_spread = group_spread(
    a1_insurance, "Insurance", minimum_n=100
)
insurance_a2_spread = group_spread(
    insurance_fairness_a2, "Insurance", minimum_n=100
)

race_a1_spread = group_spread(
    a1_race, "Race", minimum_n=100
)
race_a2_spread = group_spread(
    race_fairness_a2, "Race", minimum_n=100
)

disparity_summary = pd.DataFrame([
    {
        "Attribute": "Insurance",
        "Experiment": "A1",
        **insurance_a1_spread
    },
    {
        "Attribute": "Insurance",
        "Experiment": "A2",
        **insurance_a2_spread
    },
    {
        "Attribute": "Race",
        "Experiment": "A1",
        **race_a1_spread
    },
    {
        "Attribute": "Race",
        "Experiment": "A2",
        **race_a2_spread
    }
])

print("Subgroup disparity summary (groups with n >= 100)")
display(disparity_summary.round(4))


## 13. Summary of Findings

Laboratory feature reduction substantially decreased dimensionality without materially changing overall predictive performance.

Applying the pre-specified 5% observation-coverage criterion reduced the laboratory representation from 458 to 120 variables, corresponding to a 73.8% reduction in laboratory dimensionality. Combined with the ATC3 medication representation established in A1, the complete model feature space decreased from 5,721 predictors in A0 to 353 predictors in A2.

Relative to A1, A2 produced only minor changes in held-out performance. ROC-AUC changed from 0.8700 to 0.8691, PR-AUC from 0.7737 to 0.7728, and accuracy from 0.8225 to 0.8222. The held-out confusion matrices were also nearly identical, indicating that removal of low-coverage laboratory variables had little effect on aggregate classification behavior.

Subgroup effects were heterogeneous. Across the major insurance groups, the range in Long-stay recall and false-positive rates decreased after laboratory reduction. Across racial groups with adequate held-out representation, the Long-stay recall range increased modestly, whereas the false-positive-rate range decreased. These results do not establish an overall improvement or deterioration in fairness; rather, they indicate that the reduced laboratory representation broadly preserves the subgroup performance characteristics observed in A1.

Very small subgroups, including Pacific Islander and Multiple Race/Ethnicity categories, are reported for completeness but are not used for strong comparative conclusions because their estimates are based on limited sample sizes.

Overall, A2 provides a substantially lower-dimensional representation while retaining nearly the same predictive discrimination and subgroup-performance profile as A1. This reduced representation is therefore adopted as the candidate feature space for subsequent centralized and federated-learning experiments.


## Laboratory Feature Reduction Workflow

Laboratory feature reduction was performed using an outcome-independent observation-coverage criterion.

1. The A1 dataset was used as the starting point, containing 458 laboratory variables and the reduced ATC3 medication representation.

2. Laboratory availability was measured from the processed long-format laboratory data before zero filling, since missing laboratory measurements are represented as zero in the final No-Imputation matrix.

3. For each laboratory feature \(j\), stay-level observation coverage was calculated as:

\[
\text{Coverage}_j =
\frac{\text{number of ICU stays with at least one numeric observation for feature } j}
{\text{total number of ICU stays}}
\]

4. Several minimum-coverage thresholds were examined. A 5% threshold retained 120 laboratory variables, while a 10% threshold retained 99. Because the stricter threshold removed only 21 additional variables, several of which were clinically interpretable laboratory measurements, the 5% threshold was selected as the more conservative reduction criterion.

5. The final selection rule was therefore:

\[
\text{retain laboratory feature } j
\iff
\text{Coverage}_j \geq 0.05
\]

6. Applying this criterion reduced the laboratory feature space from:

\[
458 \rightarrow 120
\]

corresponding to a 73.8% reduction in laboratory dimensionality.

7. Only laboratory variables below the 5% coverage threshold were removed. The patient cohort, ATC3 medication representation, routine chart variables, LOS definition, patient-level splits, and downstream modeling procedure were kept unchanged.

As a result, the complete model feature space was reduced from 691 predictors in A1 to 353 predictors in A2 before model evaluation.